# Memória dinâmica

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Alocar memória cujo tamanho só se conhece durante a execução, com malloc e calloc, conferir NULL e liberá-la.
- Aumentar um buffer com realloc, dobrando a capacidade, e explicar por que o ponteiro antigo não pode mais ser usado.
- Montar arrays de linhas com tamanhos diferentes, ou um bloco só indexado por linha e coluna.

**Neste notebook**, uma ideia de cada vez (rode cada célula antes de seguir):
1. Por que pedir memória durante a execução?
2. malloc, sizeof e free
3. calloc: um bloco que começa zerado
4. realloc: aumentando um bloco
5. Memória que dura mais que a função
6. Os três bugs de memória

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check_int e check_str imprimem ✓ ou ✗; os exercícios abaixo usam
void check_int(const char *what, long actual, long expected) {
    if (actual == expected) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: esperado %ld, veio %ld\n", what, expected, actual);
    }
}

void check_str(const char *what, const char *actual, const char *expected) {
    if (actual == NULL) {
        actual = "(NULL)";
    }
    if (strcmp(actual, expected) == 0) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: esperado \"%s\", veio \"%s\"\n", what, expected, actual);
    }
}

## 1. Por que pedir memória durante a execução?

O tamanho de um array é fixo quando você escreve o código, e um array local some quando a função dele termina. As duas coisas são um problema quando o tamanho vem dos dados (quantos números o usuário vai digitar?) ou quando os dados precisam durar mais que a função. A resposta é o **heap**: uma área grande onde o programa pede blocos enquanto roda, e os devolve quando termina.

| | Stack (variáveis locais) | Heap (`malloc`) |
|---|---|---|
| tamanho | fixo no código | escolhido durante a execução |
| dura | até a função terminar | até você dar `free` |
| espaço | alguns megabytes | boa parte da memória do computador |

In [ ]:
int n = 5;                              // imagine que isso veio da entrada
int *squares = malloc(n * sizeof *squares);
if (squares == NULL) {
    return 1;                           // sem memória: desiste
}
for (int i = 0; i < n; i++) {
    squares[i] = i * i;                 // usado como um array
}
for (int i = 0; i < n; i++) {
    printf("%d ", squares[i]);
}
printf("\n");
free(squares);                          // devolve

🧪 **Experimente:** mude `n` para 12 e rode: o tamanho agora é só uma variável.

## 2. malloc, sizeof e free

`malloc(bytes)` reserva um bloco com esse número de bytes e devolve um ponteiro para ele, ou `NULL` quando não há memória. Calcule os bytes com `sizeof`: `n * sizeof *p` é "n elementos do tipo para onde `p` aponta", que continua certo mesmo se você mudar o tipo de `p`. O conteúdo do bloco começa com **lixo**. Quando terminar, `free(p)` o devolve. Os slides acompanham um bloco do `malloc` ao `free`.

#### 🎞️ malloc, realloc e free

![malloc reserva um bloco no heap, realloc o aumenta (talvez movendo-o) e free o devolve, enquanto o ponteiro guarda o endereço antigo até receber NULL](malloc-steps.pt-br.svg)

In [ ]:
double *prices = malloc(3 * sizeof *prices);
if (prices == NULL) {
    return 1;
}
printf("pedi %zu bytes\n", 3 * sizeof *prices);
prices[0] = 2.5;
prices[1] = 4.0;
prices[2] = 1.25;
double total = 0;
for (int i = 0; i < 3; i++) {
    total += prices[i];
}
printf("total %.2f\n", total);
free(prices);
prices = NULL;          // para ninguém usar o endereço antigo por engano

🧪 **Experimente:** leia `prices[0]` depois do `free` (e antes do `prices = NULL`) e rode: a verificação de memória acusa um "heap-use-after-free".

## 3. calloc: um bloco que começa zerado

`calloc(quantidade, tamanho)` é como `malloc(quantidade * tamanho)`, mas o bloco começa **cheio de zeros**. Use-o para contadores e tabelas que começam vazios.

In [ ]:
const char *text = "mississippi";
int *counts = calloc(26, sizeof *counts);
if (counts == NULL) {
    return 1;
}
for (int i = 0; text[i] != '\0'; i++) {
    counts[text[i] - 'a']++;
}
for (int c = 0; c < 26; c++) {
    if (counts[c] > 0) {
        printf("%c: %d\n", 'a' + c, counts[c]);
    }
}
free(counts);

🧪 **Experimente:** conte as letras de outra palavra. O que aconteceria com uma letra maiúscula ou um espaço? (O índice cairia fora de 0 a 25.)

### ✏️ Exercício: Uma sequência no heap

Devolva um bloco novo de n ints com 0, 1, …, n − 1. Quem chama libera.

In [ ]:
int *range_array(int n) {
    // TODO: aloque n ints com malloc e preencha
    return calloc(n, sizeof(int));
}

int main(void) {
    int *r = range_array(4);
    check_int("r[0]", r[0], 0);
    check_int("r[3]", r[3], 3);
    free(r);
    return 0;
}

<details>
<summary>Ver uma resposta</summary>

```c
int *range_array(int n) {
    int *a = malloc(n * sizeof(int));
    if (a == NULL) {
        return NULL;
    }
    for (int i = 0; i < n; i++) {
        a[i] = i;
    }
    return a;
}

int main(void) {
    int *r = range_array(4);
    check_int("r[0]", r[0], 0);
    check_int("r[3]", r[3], 3);
    free(r);
    return 0;
}
```

</details>

## 4. realloc: aumentando um bloco

Quando um bloco enche, `realloc(p, novos_bytes)` dá um maior, mantendo os valores antigos. Ele **pode mover** o bloco para outro endereço, então use sempre o ponteiro que ele devolve. Ele devolve `NULL` se falhar, deixando o bloco antigo intacto, então guarde o resultado numa variável nova antes. Uma estratégia comum é **dobrar** a capacidade a cada vez que enche, para o crescimento continuar rápido.

In [ ]:
int capacity = 2;
int count = 0;
int *values = malloc(capacity * sizeof *values);
if (values == NULL) {
    return 1;
}
for (int v = 10; v <= 70; v += 10) {
    if (count == capacity) {
        capacity *= 2;
        int *bigger = realloc(values, capacity * sizeof *values);
        if (bigger == NULL) {
            free(values);
            return 1;
        }
        values = bigger;
        printf("cresceu para %d\n", capacity);
    }
    values[count++] = v;
}
for (int i = 0; i < count; i++) {
    printf("%d ", values[i]);
}
printf("\n");
free(values);

🧪 **Experimente:** mude o laço para ir até 1000: quantas vezes ele cresce? Depois cresça de 1 em 1 em vez de dobrar, e conte as chamadas a `realloc`.

## 5. Memória que dura mais que a função

Um bloco do heap dura até ser liberado, mesmo depois que a função que o criou termina. Então uma função pode montar algo no heap e **devolver o ponteiro**. Em troca, quem chama passa a ser responsável por chamar `free`. Escreva essa regra num comentário acima da função.

In [ ]:
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

/* Devolve "palavra palavra…" (times cópias) num bloco novo. Quem chama libera. */
char *repeat(const char *word, int times) {
    size_t len = strlen(word);
    char *out = malloc(times * (len + 1) + 1);
    if (out == NULL) {
        return NULL;
    }
    out[0] = '\0';
    for (int i = 0; i < times; i++) {
        if (i > 0) {
            strcat(out, " ");
        }
        strcat(out, word);
    }
    return out;
}

int main(void) {
    char *cheer = repeat("vai", 3);
    if (cheer == NULL) {
        return 1;
    }
    printf("%s!\n", cheer);
    free(cheer);
    return 0;
}

🧪 **Experimente:** mude o `repeat` para usar um array local `char out[100];` e devolvê-lo: leia o aviso. Esse array some quando a função termina.

### ✏️ Exercício: Duplique uma string

Devolva uma cópia de s num bloco novo do heap. Lembre do byte extra do `'\0'`.

In [ ]:
char *duplicate(const char *s) {
    // TODO: aloque strlen(s) + 1 bytes e use strcpy
    return calloc(1, 1);
}

int main(void) {
    char *d = duplicate("hello");
    check_str("a cópia", d, "hello");
    char *e = duplicate("");
    check_str("uma cópia vazia", e, "");
    free(d);
    free(e);
    return 0;
}

<details>
<summary>Ver uma resposta</summary>

```c
char *duplicate(const char *s) {
    char *copy = malloc(strlen(s) + 1);
    if (copy == NULL) {
        return NULL;
    }
    strcpy(copy, s);
    return copy;
}

int main(void) {
    char *d = duplicate("hello");
    check_str("a cópia", d, "hello");
    char *e = duplicate("");
    check_str("uma cópia vazia", e, "");
    free(d);
    free(e);
    return 0;
}
```

</details>

## 6. Os três bugs de memória

- **Uso depois do free:** depois de `free(p)`, `p` ainda guarda o endereço antigo, mas o bloco não é mais seu. Ponha `p = NULL` logo depois de liberar.
- **Free duplo:** liberar o mesmo bloco duas vezes estraga o heap.
- **Vazamento:** perder o último ponteiro para um bloco sem liberá-lo. A memória fica ocupada até o programa terminar; num programa que roda por muito tempo, os vazamentos se acumulam.

As verificações de memória do Sphinx param o programa nos dois primeiros e mostram as linhas envolvidas.

In [ ]:
int *a = malloc(sizeof *a);
if (a == NULL) {
    return 1;
}
*a = 42;
printf("%d\n", *a);
free(a);
a = NULL;
if (a != NULL) {
    printf("nunca chega aqui\n");
}

🧪 **Experimente:** apague o `a = NULL;` e acrescente um segundo `free(a);`: leia o relatório de "double-free". (`free(NULL)` não faz nada, mais um motivo para pôr os ponteiros em `NULL`.)

## Em resumo

Os arrays têm um tamanho fixado quando você escreve o código. Quando o tamanho vem dos dados, peça memória durante a execução: `malloc(bytes)` devolve um ponteiro para um bloco novo no **heap**, ou `NULL` quando não há. O bloco dura até você chamar `free`, mesmo depois que a função que o criou retorna.

```c
#include <stdio.h>
#include <stdlib.h>

int main(void) {
    int capacity = 4, count = 0, value;
    int *values = malloc(capacity * sizeof *values);
    if (values == NULL) return 1;
    while (scanf("%d", &value) == 1) {
        if (count == capacity) {                 // cheio: dobra
            capacity *= 2;
            int *bigger = realloc(values, capacity * sizeof *values);
            if (bigger == NULL) { free(values); return 1; }
            values = bigger;
        }
        values[count++] = value;
    }
    printf("%d numbers\n", count);
    free(values);
    return 0;
}
```

O `realloc` pode mover o bloco, então continue usando o ponteiro que ele devolve.

**Cuidado:** depois de `free(p)`, `p` ainda guarda o endereço antigo. Usá-lo, ou liberá-lo duas vezes, é comportamento indefinido; as verificações de memória pegam isso.

## Leituras

- [Manual Memory Allocation](https://beej.us/guide/bgc/html/split/manual-memory-allocation.html) (Beej's Guide to C Programming, chapter 12, 12 min): malloc e free, conferência de erros, calloc e realloc, inclusive "Reading in Lines of Arbitrary Length", que é o desafio Longest Line quase linha por linha.
- [Lecture 4: Copying and malloc](https://cs50.harvard.edu/x/2026/notes/4/#copying-and-malloc) (CS50x (Harvard), lecture notes, 10 min): Opcional: de "Copying and malloc" até "Garbage Values": por que copiar um ponteiro não copia a string, e a pilha e o heap numa figura.

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Quantidade desconhecida** · Fácil
- **Linha mais longa** · Médio
- **Triângulo de Pascal** · Médio
- **Multiplicação de matrizes** · Difícil